In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv('C:\\Users\\Abdullah\\Desktop\\Code\\Sentiment Based Stock\\Sentiment Based Stocks\\results\\final_dataset_labeled.csv')

In [3]:
price_cols = ['open', 'high', 'low', 'close', 'volume',
              'returns', 'volatility', 'rsi', 'ma_7', 'ma_30']

TARGET = 'label_3h'

In [5]:
print(df['hour'].min())
print(df['hour'].max())

2024-07-06 04:00:00
2025-09-14 17:00:00


In [7]:
train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 2146
Test size: 1345


In [8]:
scaler = StandardScaler()
train_X = scaler.fit_transform(train[price_cols])
test_X  = scaler.transform(test[price_cols])

train_y = train[TARGET].values
test_y  = test[TARGET].values

In [9]:
SEQ_LEN = 24

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = []
        self.y = []
        for i in range(SEQ_LEN, len(X)):
            self.X.append(X[i-SEQ_LEN:i])
            self.y.append(y[i])
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = SeqDataset(train_X, train_y)
test_ds  = SeqDataset(test_X,  test_y)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [10]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last timestep
        return self.fc(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = BiLSTM(input_size=len(price_cols)).to(device)

Device: cuda


In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

Epoch 5/30 — Loss: 0.6923
Epoch 10/30 — Loss: 0.6810
Epoch 15/30 — Loss: 0.6670
Epoch 20/30 — Loss: 0.6497
Epoch 25/30 — Loss: 0.6285
Epoch 30/30 — Loss: 0.6048


In [12]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        probs   = model(X_batch).cpu().numpy()
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

In [13]:
print("\n── Price-only BiLSTM Results (3h) ──")
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")


── Price-only BiLSTM Results (3h) ──
Accuracy  : 0.4883
Precision : 0.4991
Recall    : 0.4044
F1        : 0.4468
AUC       : 0.4968


In [14]:
TARGET = 'label_6h'

In [15]:
df['hour'] = pd.to_datetime(df['hour'])
train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 2146
Test size: 1345


In [16]:
scaler = StandardScaler()
train_X = scaler.fit_transform(train[price_cols])
test_X  = scaler.transform(test[price_cols])

train_y = train[TARGET].values
test_y  = test[TARGET].values

In [17]:
SEQ_LEN = 24

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = []
        self.y = []
        for i in range(SEQ_LEN, len(X)):
            self.X.append(X[i-SEQ_LEN:i])
            self.y.append(y[i])
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = SeqDataset(train_X, train_y)
test_ds  = SeqDataset(test_X,  test_y)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [18]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last timestep
        return self.fc(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = BiLSTM(input_size=len(price_cols)).to(device)

Device: cuda


In [19]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

Epoch 5/30 — Loss: 0.6858
Epoch 10/30 — Loss: 0.6719
Epoch 15/30 — Loss: 0.6561
Epoch 20/30 — Loss: 0.6170
Epoch 25/30 — Loss: 0.5701
Epoch 30/30 — Loss: 0.5822


In [20]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        probs   = model(X_batch).cpu().numpy()
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

In [21]:
print("\n── Price-only BiLSTM Results (6h) ──")
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")


── Price-only BiLSTM Results (6h) ──
Accuracy  : 0.4852
Precision : 0.5339
Recall    : 0.0916
F1        : 0.1563
AUC       : 0.5210
